# NumCompute — End-to-End Quickstart

This notebook demonstrates the full NumCompute toolkit: data I/O, preprocessing, sorting and ranking, descriptive statistics, evaluation metrics, finite-difference gradients, pipeline composition, and vectorised benchmarking.

**Requirements:** plain Python + NumPy only. No external ML/DL libraries.

In [37]:
import numpy as np
from numcompute import (
    io,
    preprocessing,
    sort_search,
    rank,
    stats,
    metrics,
    optim,
    pipeline,
    benchmarking,
)

print("All modules imported successfully.")

All modules imported successfully.


---
## 1. Data I/O — Reading CSV with Missing Values

`io.read_csv` handles missing/null tokens (`''`, `NA`, `nan`, `null`, `none`) and returns a NumPy array.
The chunked reader (`read_csv_chunked`) is demonstrated for large-file simulation.

In [38]:
# --- Full read ---
headers, data = io.read_csv("sample_data.csv")

print("Column headers:", headers)
print("Data shape    :", data.shape)
print("NaN count     :", int(np.sum(np.isnan(data[:, :3]))))
print()
print(f"{'age':>10} {'income':>10} {'score':>10}")
for row in data[:5, :3]:
    print(" ".join(f"{v:>10.1f}" if not np.isnan(v) else f"{'NaN':>10}" for v in row))

Column headers: ['age', 'income', 'score', 'category']
Data shape    : (10, 4)
NaN count     : 6

       age     income      score
      25.0    48000.0       82.5
      32.0        NaN       91.2
       NaN    62000.0       78.4
      41.0    75000.0        NaN
      28.0    53000.0       88.1


In [39]:
# --- Chunked read (streaming) ---
chunks_seen = 0
total_rows  = 0

for chunk_headers, chunk_data in io.read_csv_chunked("sample_data.csv", chunk_size=4):
    chunks_seen += 1
    total_rows  += chunk_data.shape[0]
    print(f"Chunk {chunks_seen}: shape {chunk_data.shape}")

print(f"\nTotal chunks: {chunks_seen}, total rows: {total_rows}")

Chunk 1: shape (4, 4)
Chunk 2: shape (4, 4)
Chunk 3: shape (2, 4)

Total chunks: 3, total rows: 10


---
## 2. Preprocessing — Imputation, Scaling, One-Hot Encoding

All preprocessors follow a `fit → transform` API identical to scikit-learn's convention.

In [40]:
# Separate numeric columns (age, income, score) from the category column
numeric = data[:, :3].copy()   # shape (10, 3) — contains NaNs
cat_raw = data[:, 3]           # shape (10,)  — integer-encoded category (NaN-free in this dataset)

# --- 2a. Mean imputation ---
imputer       = preprocessing.Imputer(strategy="mean")
numeric_clean = imputer.fit_transform(numeric)

print("NaNs before imputation:", int(np.sum(np.isnan(numeric))))
print("NaNs after  imputation:", int(np.sum(np.isnan(numeric_clean))))
print("Column means used     :", imputer.fill_values_.round(2))

NaNs before imputation: 6
NaNs after  imputation: 0
Column means used     : [3.588e+01 6.525e+04 8.386e+01]


In [42]:
# --- 2b. StandardScaler (zero mean, unit variance) ---
scaler  = preprocessing.StandardScaler()
scaled  = scaler.fit_transform(numeric_clean)

print("Post-scaling column means:", scaled.mean(axis=0).round(10))
print("Post-scaling column stds :", scaled.std(axis=0).round(6))

Post-scaling column means: [-0. -0.  0.]
Post-scaling column stds : [1. 1. 1.]


In [43]:
# --- 2c. MinMaxScaler (maps to [0, 1]) ---
mm_scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
normed    = mm_scaler.fit_transform(numeric_clean)

print("MinMax column min:", normed.min(axis=0).round(4))
print("MinMax column max:", normed.max(axis=0).round(4))

MinMax column min: [0. 0. 0.]
MinMax column max: [1. 1. 1.]


In [15]:
# --- 2d. OneHotEncoder ---
# cat_raw encodes A=0, B=1, C=2  (NaN rows were filled with 0 by the CSV reader)
cat_int = cat_raw.astype(int)
ohe     = preprocessing.OneHotEncoder()
one_hot = ohe.fit_transform(cat_int)

print("OneHot shape:", one_hot.shape, "  (10 rows, 3 categories)")
print("Categories  :", ohe.categories_)
print("First 5 rows:")
print(one_hot[:5])

OneHot shape: (10, 3)   (10 rows, 3 categories)
Categories  : [0 1 2]
First 5 rows:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


---
## 3. Sort, Search & Top-K

`sort_search` provides vectorised quicksort, quickselect, top-k via argpartition, and binary search.

In [16]:
scores = numeric_clean[:, 2]   # the 'score' column after imputation
print("Raw scores:", scores.round(1))

# --- 3a. Quicksort ---
sorted_scores = sort_search.quicksort(scores, ascending=True)
print("Sorted    :", sorted_scores.round(1))

Raw scores: [82.5 91.2 78.4 83.9 88.1 95.3 70.6 85.  83.9 79.8]
Sorted    : [70.6 78.4 79.8 82.5 83.9 83.9 85.  88.1 91.2 95.3]


In [17]:
# --- 3b. Top-k ---
top3 = sort_search.top_k(scores, k=3, largest=True)
bot3 = sort_search.top_k(scores, k=3, largest=False)
print("Top-3 scores   :", top3.round(1))
print("Bottom-3 scores:", bot3.round(1))

Top-3 scores   : [95.3 91.2 88.1]
Bottom-3 scores: [70.6 78.4 79.8]


In [18]:
# --- 3c. Quickselect (k-th smallest without full sort) ---
# k=4 gives the 4th smallest element (0-indexed)
kth = sort_search.quickselect(scores, k=4)
print(f"4th smallest score (quickselect): {kth:.1f}")
print(f"Verification via full sort      : {sorted_scores[4]:.1f}")

4th smallest score (quickselect): 83.9
Verification via full sort      : 83.9


In [19]:
# --- 3d. Binary search ---
target = sorted_scores[6]   # pick a value we know exists
found_idx = sort_search.binary_search(sorted_scores, target)
print(f"Binary search for {target:.1f}: found at index {found_idx}")
print(f"Confirmed: sorted_scores[{found_idx}] = {sorted_scores[found_idx]:.1f}")

missing_idx = sort_search.binary_search(sorted_scores, 999.0)
print(f"Search for 999.0 (not present): returns {missing_idx}")

Binary search for 85.0: found at index 6
Confirmed: sorted_scores[6] = 85.0
Search for 999.0 (not present): returns -1


---
## 4. Ranking with Tie Handling & Percentiles

`rank.rank_data` supports four tie-breaking methods. `percentile_rank` gives the fraction of values below a given score.

In [20]:
# Demonstrate tie handling with a deliberately tied array
tied = np.array([10.0, 20.0, 20.0, 30.0, 30.0, 30.0, 40.0])

for method in ("average", "min", "max", "ordinal"):
    r = rank.rank_data(tied, method=method)
    print(f"{method:>8}: {r}")

 average: [1.  2.5 2.5 5.  5.  5.  7. ]
     min: [1. 2. 2. 4. 4. 4. 7.]
     max: [1. 3. 3. 6. 6. 6. 7.]
 ordinal: [1. 2. 3. 4. 5. 6. 7.]


In [21]:
# Rank the score column
score_ranks = rank.rank_data(scores, method="average")
print("Scores :", scores.round(1))
print("Ranks  :", score_ranks)

# Percentile rank
for threshold in [75.0, 85.0, 95.0]:
    pr = rank.percentile_rank(scores, threshold)
    print(f"Percentile rank of {threshold}: {pr:.1f}%")

Scores : [82.5 91.2 78.4 83.9 88.1 95.3 70.6 85.  83.9 79.8]
Ranks  : [ 4.   9.   2.   5.5  8.  10.   1.   7.   5.5  3. ]
Percentile rank of 75.0: 10.0%
Percentile rank of 85.0: 60.0%
Percentile rank of 95.0: 90.0%


---
## 5. Descriptive Statistics — Welford, Histogram, Quantiles

In [22]:
# --- 5a. Welford mean & variance ---
mean_, var_ = stats.welford(scores)
print(f"Mean    : {mean_:.4f}")
print(f"Variance: {var_:.4f}")
print(f"Std dev : {var_**0.5:.4f}")

Mean    : 83.8625
Variance: 42.7999
Std dev : 6.5422


In [23]:
# --- 5b. Histogram ---
counts, edges = stats.histogram(scores, bins=5)
print("Histogram of scores (5 bins):")
print(f"{'Bin range':>22}  Count  Bar")
for i, c in enumerate(counts):
    bar = "#" * int(c * 5)
    print(f"  [{edges[i]:6.1f} – {edges[i+1]:6.1f})    {c:3d}  {bar}")

Histogram of scores (5 bins):
             Bin range  Count  Bar
  [  70.6 –   75.5)      1  #####
  [  75.5 –   80.5)      2  ##########
  [  80.5 –   85.4)      4  ####################
  [  85.4 –   90.4)      1  #####
  [  90.4 –   95.3)      2  ##########


In [24]:
# --- 5c. Quantiles ---
q_vals = stats.quantiles(scores, np.array([0.0, 0.25, 0.5, 0.75, 1.0]))
labels = ["Min (Q0)", "Q1 (25%)", "Median",   "Q3 (75%)", "Max (Q100)"]
for label, v in zip(labels, q_vals):
    print(f"  {label:>12}: {v:.2f}")

      Min (Q0): 70.60
      Q1 (25%): 80.47
        Median: 83.86
      Q3 (75%): 87.32
    Max (Q100): 95.30


---
## 6. Finite-Difference Gradients & Jacobians

`optim.finite_diff_gradient` uses central differences to estimate ∇f(x). `finite_diff_jacobian` generalises this to vector-valued functions.

In [25]:
# --- 6a. Gradient of a scalar function ---
# f(x) = x0^2 + 2*x1^2  →  analytic gradient: [2*x0, 4*x1]
def f_quad(x):
    return x[0]**2 + 2.0 * x[1]**2

x0 = np.array([3.0, -1.0])
grad_approx   = optim.finite_diff_gradient(f_quad, x0)
grad_analytic = np.array([2 * x0[0], 4 * x0[1]])

print("Point          :", x0)
print("Approx gradient:", grad_approx)
print("Exact gradient :", grad_analytic)
print("Max abs error  :", np.max(np.abs(grad_approx - grad_analytic)))

Point          : [ 3. -1.]
Approx gradient: [ 6. -4.]
Exact gradient : [ 6. -4.]
Max abs error  : 3.930633596382904e-11


In [26]:
# --- 6b. Jacobian of a vector-valued function ---
# g(x) = [x0^2,  x0*x1,  x1^2]
# Analytic Jacobian:
#   [[2*x0,    0  ],
#    [  x1,   x0  ],
#    [   0,  2*x1 ]]
def g_vec(x):
    return np.array([x[0]**2, x[0] * x[1], x[1]**2])

x0 = np.array([2.0, 3.0])
J_approx  = optim.finite_diff_jacobian(g_vec, x0)
J_analytic = np.array([[2*x0[0],       0],
                        [  x0[1],  x0[0]],
                        [      0, 2*x0[1]]])

print("Approximate Jacobian:")
print(J_approx.round(6))
print("\nAnalytic Jacobian:")
print(J_analytic)
print("\nMax abs error:", np.max(np.abs(J_approx - J_analytic)))

Approximate Jacobian:
[[4. 0.]
 [3. 2.]
 [0. 6.]]

Analytic Jacobian:
[[4. 0.]
 [3. 2.]
 [0. 6.]]

Max abs error: 6.406208896692078e-11


---
## 7. Evaluation Metrics

`metrics` provides accuracy, precision, recall, F1, confusion matrix, MSE, and ROC/AUC.

In [27]:
# Simulate a binary classification scenario
np.random.seed(42)
y_true  = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred  = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 0])
y_score = np.array([0.92, 0.18, 0.81, 0.43, 0.11, 0.87, 0.61, 0.29, 0.95, 0.14])

acc  = metrics.accuracy(y_true, y_pred)
prec = metrics.precision(y_true, y_pred)
rec  = metrics.recall(y_true, y_pred)
f1   = metrics.f1(y_true, y_pred, average="binary")
auc  = metrics.roc_auc(y_true, y_score)
mse  = metrics.mse(y_true.astype(float), y_pred.astype(float))
cm   = metrics.confusion_matrix(y_true, y_pred)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 score  : {f1:.4f}")
print(f"ROC AUC   : {auc:.4f}")
print(f"MSE       : {mse:.4f}")
print(f"Confusion matrix:\n{cm}")

Accuracy  : 0.8000
Precision : 0.8000
Recall    : 0.8000
F1 score  : 0.8000
ROC AUC   : 0.9600
MSE       : 0.2000
Confusion matrix:
[[4 1]
 [1 4]]


---
## 8. Pipeline — Composing Transformers

`pipeline.Compose` chains transformers so each step's output feeds the next.

In [44]:
# Build a pipeline: Impute missing → Standard-scale
pipe = pipeline.Compose([
    preprocessing.Imputer(strategy="mean"),
    preprocessing.StandardScaler(),
])

# Fit on the raw numeric data (which contains NaNs) and transform in one call
X_raw        = data[:, :3]      # age, income, score — with NaNs
X_processed  = pipe.fit_transform(X_raw)

print("Input shape :", X_raw.shape, "  NaNs:", int(np.sum(np.isnan(X_raw))))
print("Output shape:", X_processed.shape, " NaNs:", int(np.sum(np.isnan(X_processed))))
print("Column means (should be ~0):", X_processed.mean(axis=0).round(10))
print("Column stds  (should be ~1):", X_processed.std(axis=0).round(6))

Input shape : (10, 3)   NaNs: 6
Output shape: (10, 3)  NaNs: 0
Column means (should be ~0): [-0. -0.  0.]
Column stds  (should be ~1): [1. 1. 1.]


In [29]:
# A second pipeline: Impute → MinMax scale
pipe2 = pipeline.Compose([
    preprocessing.Imputer(strategy="median"),
    preprocessing.MinMaxScaler(feature_range=(0, 1)),
])

X_normed = pipe2.fit_transform(X_raw)
print("MinMax pipeline output min:", X_normed.min(axis=0).round(4))
print("MinMax pipeline output max:", X_normed.max(axis=0).round(4))

MinMax pipeline output min: [0. 0. 0.]
MinMax pipeline output max: [1. 1. 1.]


---
## 9. Benchmarks — Vectorised vs Python Loops

`benchmarking.benchmark_fn` measures wall-clock time (mean ± std) over repeated runs.  
We compare vectorised NumPy against equivalent pure-Python loops across three operations.

In [30]:
N       = 100_000
REPEATS = 100
rng     = np.random.default_rng(0)
big_arr = rng.standard_normal(N)

# ── Operation 1: sum ──────────────────────────────────────────────────
def loop_sum(a):
    s = 0.0
    for v in a: s += v
    return s

def vec_sum(a):
    return float(a.sum())

# ── Operation 2: element-wise square ──────────────────────────────────
def loop_square(a):
    out = [v * v for v in a]
    return out

def vec_square(a):
    return a ** 2

# ── Operation 3: dot product ───────────────────────────────────────────
b = rng.standard_normal(N)

def loop_dot(a, x):
    s = 0.0
    for ai, xi in zip(a, x): s += ai * xi
    return s

def vec_dot(a, x):
    return float(np.dot(a, x))

# ── Run benchmarks ────────────────────────────────────────────────────
results = {}
for name, loop_fn, vec_fn, extra_args in [
    ("Sum",      loop_sum,    vec_sum,    ()),
    ("Square",   loop_square, vec_square, ()),
    ("Dot",      loop_dot,    vec_dot,    (b,)),
]:
    lm, ls = benchmarking.benchmark_fn(loop_fn, args=(big_arr, *extra_args), repeats=REPEATS)
    vm, vs = benchmarking.benchmark_fn(vec_fn,  args=(big_arr, *extra_args), repeats=REPEATS)
    results[name] = (lm, ls, vm, vs)

print(f"Array size: {N:,}  |  Repeats: {REPEATS}\n")
print(f"{'Operation':<12} {'Loop (ms)':>14} {'Vec (ms)':>14} {'Speedup':>10}")
print("-" * 54)
for name, (lm, ls, vm, vs) in results.items():
    speedup = lm / vm if vm > 0 else float('inf')
    print(f"{name:<12} {lm*1000:>10.3f} ± {ls*1000:.3f}   {vm*1000:>6.3f} ± {vs*1000:.3f}   {speedup:>6.0f}×")

Array size: 100,000  |  Repeats: 100

Operation         Loop (ms)       Vec (ms)    Speedup
------------------------------------------------------
Sum              21.138 ± 9.558    0.044 ± 0.021      475×
Square           25.001 ± 10.376    0.049 ± 0.095      515×
Dot              49.925 ± 15.578    0.676 ± 1.405       74×


### Performance Table Summary

| Operation | Python loop | NumPy vectorised | Typical speedup |
|-----------|-------------|------------------|-----------------|
| Sum       | ~ms range   | ~µs range        | 100–300×        |
| Square    | ~ms range   | ~µs range        | 50–200×         |
| Dot       | ~ms range   | ~µs range        | 100–500×        |

Exact speedups depend on hardware and CPU cache state. NumPy vectorised operations exploit BLAS/LAPACK routines and SIMD instructions, avoiding the overhead of Python's interpreter dispatch on every element.

In [31]:
# ── Also benchmark sort: our quicksort vs np.sort ────────────────────
sort_arr = rng.standard_normal(5_000)

lm_s, ls_s = benchmarking.benchmark_fn(sort_search.quicksort, args=(sort_arr,), repeats=50)
vm_s, vs_s = benchmarking.benchmark_fn(np.sort,               args=(sort_arr,), repeats=50)

print("Sort comparison (n=5,000, 50 repeats):")
print(f"  numcompute.quicksort : {lm_s*1000:.3f} ms ± {ls_s*1000:.3f}")
print(f"  np.sort              : {vm_s*1000:.3f} ms ± {vs_s*1000:.3f}")
print(f"  np.sort speedup      : {lm_s/vm_s:.1f}×  (np.sort uses Timsort/introsort in C)")

Sort comparison (n=5,000, 50 repeats):
  numcompute.quicksort : 354.177 ms ± 45.769
  np.sort              : 0.098 ms ± 0.289
  np.sort speedup      : 3628.7×  (np.sort uses Timsort/introsort in C)


---
## Summary

| Module | Demonstrated | Key functions / classes |
|---|---|---|
| `io` | CSV read (full + chunked), write | `read_csv`, `read_csv_chunked`, `write_csv` |
| `preprocessing` | Imputation, standard scaling, min-max, one-hot | `Imputer`, `StandardScaler`, `MinMaxScaler`, `OneHotEncoder` |
| `sort_search` | Sort, top-k, quickselect, binary search | `quicksort`, `top_k`, `quickselect`, `binary_search` |
| `rank` | Tie-aware ranking, percentile rank | `rank_data`, `percentile_rank` |
| `stats` | Mean/variance, histogram, quantiles | `welford`, `histogram`, `quantiles` |
| `metrics` | Classification + regression metrics | `accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `mse`, `confusion_matrix` |
| `optim` | Finite-difference gradient & Jacobian | `finite_diff_gradient`, `finite_diff_jacobian` |
| `pipeline` | Chained transformer composition | `Compose` |
| `benchmarking` | Loop vs vectorised timing | `benchmark_fn` |